<a href="https://colab.research.google.com/github/kasugy/codigos-de-ejemplos/blob/main/generador%20de%20proyectos%20fullstackjava.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# -*- coding: utf-8 -*-
"""Generador_Proyecto_FullStack_Java_React.ipynb
Genera un proyecto completo Full Stack (Spring Boot + React) listo para descargar.
Autor: Carolina Ulloa
"""

import os
import zipfile
from google.colab import files

# Crear carpeta raíz del proyecto
PROJECT_NAME = "fullstack-todo-app"
os.makedirs(PROJECT_NAME, exist_ok=True)

# ========================
# BACKEND: Spring Boot
# ========================
BACKEND_DIR = os.path.join(PROJECT_NAME, "backend")
os.makedirs(BACKEND_DIR, exist_ok=True)

# 1. pom.xml
pom_content = '''<?xml version="1.0" encoding="UTF-8"?>
<project xmlns="http://maven.apache.org/POM/4.0.0"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd">
    <modelVersion>4.0.0</modelVersion>
    <parent>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-parent</artifactId>
        <version>3.1.5</version>
        <relativePath/>
    </parent>
    <groupId>com.example</groupId>
    <artifactId>todo-backend</artifactId>
    <version>0.0.1-SNAPSHOT</version>
    <name>todo-backend</name>
    <description>Backend API para To-Do app</description>

    <properties>
        <java.version>17</java.version>
    </properties>

    <dependencies>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-web</artifactId>
        </dependency>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-data-jpa</artifactId>
        </dependency>
        <dependency>
            <groupId>com.h2database</groupId>
            <artifactId>h2</artifactId>
            <scope>runtime</scope>
        </dependency>
        <dependency>
            <groupId>org.projectlombok</groupId>
            <artifactId>lombok</artifactId>
            <optional>true</optional>
        </dependency>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-test</artifactId>
            <scope>test</scope>
        </dependency>
    </dependencies>

    <build>
        <plugins>
            <plugin>
                <groupId>org.springframework.boot</groupId>
                <artifactId>spring-boot-maven-plugin</artifactId>
                <configuration>
                    <excludes>
                        <exclude>
                            <groupId>org.projectlombok</groupId>
                            <artifactId>lombok</artifactId>
                        </exclude>
                    </excludes>
                </configuration>
            </plugin>
        </plugins>
    </build>
</project>
'''
with open(os.path.join(BACKEND_DIR, "pom.xml"), "w") as f:
    f.write(pom_content)

# 2. Estructura de paquetes
src_main_java = os.path.join(BACKEND_DIR, "src", "main", "java", "com", "example", "todo")
os.makedirs(src_main_java, exist_ok=True)
src_resources = os.path.join(BACKEND_DIR, "src", "main", "resources")
os.makedirs(src_resources, exist_ok=True)

# 3. Application main
main_app = '''package com.example.todo;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class TodoApplication {
    public static void main(String[] args) {
        SpringApplication.run(TodoApplication.class, args);
    }
}
'''
with open(os.path.join(src_main_java, "TodoApplication.java"), "w") as f:
    f.write(main_app)

# 4. Entidad Task
entity_dir = os.path.join(src_main_java, "model")
os.makedirs(entity_dir, exist_ok=True)
entity_code = '''package com.example.todo.model;

import jakarta.persistence.*;
import lombok.AllArgsConstructor;
import lombok.Data;
import lombok.NoArgsConstructor;

@Entity
@Table(name = "tasks")
@Data
@NoArgsConstructor
@AllArgsConstructor
public class Task {
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;
    private String title;
    private boolean completed;
}
'''
with open(os.path.join(entity_dir, "Task.java"), "w") as f:
    f.write(entity_code)

# 5. Repository
repo_dir = os.path.join(src_main_java, "repository")
os.makedirs(repo_dir, exist_ok=True)
repo_code = '''package com.example.todo.repository;

import com.example.todo.model.Task;
import org.springframework.data.jpa.repository.JpaRepository;
import org.springframework.stereotype.Repository;

@Repository
public interface TaskRepository extends JpaRepository<Task, Long> {
}
'''
with open(os.path.join(repo_dir, "TaskRepository.java"), "w") as f:
    f.write(repo_code)

# 6. Service
service_dir = os.path.join(src_main_java, "service")
os.makedirs(service_dir, exist_ok=True)
service_code = '''package com.example.todo.service;

import com.example.todo.model.Task;
import com.example.todo.repository.TaskRepository;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.stereotype.Service;
import java.util.List;

@Service
public class TaskService {
    @Autowired
    private TaskRepository taskRepository;

    public List<Task> getAllTasks() {
        return taskRepository.findAll();
    }

    public Task createTask(Task task) {
        return taskRepository.save(task);
    }

    public Task updateTask(Long id, Task taskDetails) {
        Task task = taskRepository.findById(id).orElseThrow();
        task.setTitle(taskDetails.getTitle());
        task.setCompleted(taskDetails.isCompleted());
        return taskRepository.save(task);
    }

    public void deleteTask(Long id) {
        taskRepository.deleteById(id);
    }
}
'''
with open(os.path.join(service_dir, "TaskService.java"), "w") as f:
    f.write(service_code)

# 7. Controller
controller_dir = os.path.join(src_main_java, "controller")
os.makedirs(controller_dir, exist_ok=True)
controller_code = '''package com.example.todo.controller;

import com.example.todo.model.Task;
import com.example.todo.service.TaskService;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.web.bind.annotation.*;
import java.util.List;

@RestController
@RequestMapping("/api/tasks")
@CrossOrigin(origins = "http://localhost:5173")
public class TaskController {
    @Autowired
    private TaskService taskService;

    @GetMapping
    public List<Task> getAllTasks() {
        return taskService.getAllTasks();
    }

    @PostMapping
    public Task createTask(@RequestBody Task task) {
        return taskService.createTask(task);
    }

    @PutMapping("/{id}")
    public Task updateTask(@PathVariable Long id, @RequestBody Task task) {
        return taskService.updateTask(id, task);
    }

    @DeleteMapping("/{id}")
    public void deleteTask(@PathVariable Long id) {
        taskService.deleteTask(id);
    }
}
'''
with open(os.path.join(controller_dir, "TaskController.java"), "w") as f:
    f.write(controller_code)

# 8. application.properties
app_props = '''spring.application.name=todo-backend
server.port=8080
spring.datasource.url=jdbc:h2:mem:testdb
spring.datasource.driverClassName=org.h2.Driver
spring.datasource.username=sa
spring.datasource.password=
spring.jpa.database-platform=org.hibernate.dialect.H2Dialect
spring.h2.console.enabled=true
spring.jpa.hibernate.ddl-auto=update
'''
with open(os.path.join(src_resources, "application.properties"), "w") as f:
    f.write(app_props)

# ========================
# FRONTEND: React + Vite
# ========================
FRONTEND_DIR = os.path.join(PROJECT_NAME, "frontend")
os.makedirs(FRONTEND_DIR, exist_ok=True)

# 1. package.json
package_json = '''{
  "name": "todo-frontend",
  "private": true,
  "version": "0.0.0",
  "type": "module",
  "scripts": {
    "dev": "vite",
    "build": "vite build",
    "preview": "vite preview"
  },
  "dependencies": {
    "axios": "^1.6.0",
    "react": "^18.2.0",
    "react-dom": "^18.2.0"
  },
  "devDependencies": {
    "@types/react": "^18.2.37",
    "@types/react-dom": "^18.2.15",
    "@vitejs/plugin-react": "^4.0.0",
    "vite": "^4.4.5"
  }
}
'''
with open(os.path.join(FRONTEND_DIR, "package.json"), "w") as f:
    f.write(package_json)

# 2. index.html
index_html = '''<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>To-Do App</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.jsx"></script>
  </body>
</html>
'''
with open(os.path.join(FRONTEND_DIR, "index.html"), "w") as f:
    f.write(index_html)

# 3. vite.config.js
vite_config = '''import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'

export default defineConfig({
  plugins: [react()],
  server: {
    port: 5173
  }
})
'''
with open(os.path.join(FRONTEND_DIR, "vite.config.js"), "w") as f:
    f.write(vite_config)

# 4. Carpeta src
src_dir = os.path.join(FRONTEND_DIR, "src")
os.makedirs(src_dir, exist_ok=True)

# 5. main.jsx
main_jsx = '''import React from 'react'
import ReactDOM from 'react-dom/client'
import App from './App.jsx'
import './index.css'

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <App />
  </React.StrictMode>,
)
'''
with open(os.path.join(src_dir, "main.jsx"), "w") as f:
    f.write(main_jsx)

# 6. index.css
index_css = '''body {
  font-family: Arial, sans-serif;
  margin: 0;
  padding: 20px;
  background-color: #f0f2f5;
}
'''
with open(os.path.join(src_dir, "index.css"), "w") as f:
    f.write(index_css)

# 7. App.jsx (componente principal)
app_jsx = '''import React, { useState, useEffect } from 'react';
import axios from 'axios';
import './App.css';

const API_URL = 'http://localhost:8080/api/tasks';

function App() {
  const [tasks, setTasks] = useState([]);
  const [newTaskTitle, setNewTaskTitle] = useState('');

  useEffect(() => {
    fetchTasks();
  }, []);

  const fetchTasks = async () => {
    try {
      const response = await axios.get(API_URL);
      setTasks(response.data);
    } catch (error) {
      console.error('Error fetching tasks:', error);
    }
  };

  const addTask = async () => {
    if (!newTaskTitle.trim()) return;
    try {
      const response = await axios.post(API_URL, { title: newTaskTitle, completed: false });
      setTasks([...tasks, response.data]);
      setNewTaskTitle('');
    } catch (error) {
      console.error('Error adding task:', error);
    }
  };

  const toggleComplete = async (task) => {
    try {
      const updated = { ...task, completed: !task.completed };
      const response = await axios.put(`${API_URL}/${task.id}`, updated);
      setTasks(tasks.map(t => t.id === task.id ? response.data : t));
    } catch (error) {
      console.error('Error updating task:', error);
    }
  };

  const deleteTask = async (id) => {
    try {
      await axios.delete(`${API_URL}/${id}`);
      setTasks(tasks.filter(t => t.id !== id));
    } catch (error) {
      console.error('Error deleting task:', error);
    }
  };

  return (
    <div className="container">
      <h1>📋 To-Do List</h1>
      <div className="add-task">
        <input
          type="text"
          value={newTaskTitle}
          onChange={(e) => setNewTaskTitle(e.target.value)}
          placeholder="Nueva tarea..."
          onKeyPress={(e) => e.key === 'Enter' && addTask()}
        />
        <button onClick={addTask}>Agregar</button>
      </div>
      <ul>
        {tasks.map(task => (
          <li key={task.id} className={task.completed ? 'completed' : ''}>
            <span onClick={() => toggleComplete(task)}>{task.title}</span>
            <button onClick={() => deleteTask(task.id)}>🗑️</button>
          </li>
        ))}
      </ul>
    </div>
  );
}

export default App;
'''
with open(os.path.join(src_dir, "App.jsx"), "w") as f:
    f.write(app_jsx)

# 8. App.css
app_css = '''.container {
  max-width: 500px;
  margin: 0 auto;
  background: white;
  padding: 20px;
  border-radius: 8px;
  box-shadow: 0 2px 4px rgba(0,0,0,0.1);
}
h1 { text-align: center; }
.add-task {
  display: flex;
  gap: 10px;
  margin-bottom: 20px;
}
.add-task input {
  flex: 1;
  padding: 8px;
  border: 1px solid #ccc;
  border-radius: 4px;
}
.add-task button {
  padding: 8px 16px;
  background-color: #28a745;
  color: white;
  border: none;
  border-radius: 4px;
  cursor: pointer;
}
ul {
  list-style: none;
  padding: 0;
}
li {
  display: flex;
  justify-content: space-between;
  align-items: center;
  padding: 8px;
  border-bottom: 1px solid #eee;
}
li span {
  cursor: pointer;
  flex: 1;
}
li.completed span {
  text-decoration: line-through;
  color: #888;
}
li button {
  background: none;
  border: none;
  cursor: pointer;
  font-size: 18px;
}
'''
with open(os.path.join(src_dir, "App.css"), "w") as f:
    f.write(app_css)

# ========================
# README principal
# ========================
readme_content = '''# Full Stack To-Do App (Java Spring Boot + React)

Esta aplicación fue generada automáticamente como base para tu portafolio. Demuestra habilidades full stack.

## Tecnologías
- Backend: Spring Boot (Java 17), Maven, H2 Database
- Frontend: React, Vite, Axios

## Cómo ejecutar

### Backend
1. Abrir el proyecto `backend` en IntelliJ IDEA o VS Code con extensiones Java.
2. Ejecutar la clase `TodoApplication.java`.
3. La API estará en `http://localhost:8080/api/tasks`

### Frontend
1. Abrir terminal en la carpeta `frontend`.
2. Ejecutar `npm install`
3. Ejecutar `npm run dev`
4. Abrir `http://localhost:5173`

## Mejoras posibles
- Conectar a MySQL/PostgreSQL en lugar de H2.
- Agregar autenticación JWT.
- Desplegar en Render + Vercel.

## Contacto
GitHub: https://github.com/kasugy
LinkedIn: https://www.linkedin.com/in/carolina-ulloa-gonz%C3%A1lez-31b515159/
'''
with open(os.path.join(PROJECT_NAME, "README.md"), "w") as f:
    f.write(readme_content)

# ========================
# Empaquetar en ZIP
# ========================
zip_path = f"{PROJECT_NAME}.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(PROJECT_NAME):
        for file in filenames:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=os.path.dirname(PROJECT_NAME))
            zipf.write(file_path, arcname)

# Descargar el ZIP
files.download(zip_path)

print("✅ Proyecto generado y descargado exitosamente.")
print("Descomprime el archivo y abre la carpeta 'fullstack-todo-app'.")
print("Sigue las instrucciones del README.md para ejecutar backend y frontend.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Proyecto generado y descargado exitosamente.
Descomprime el archivo y abre la carpeta 'fullstack-todo-app'.
Sigue las instrucciones del README.md para ejecutar backend y frontend.
